# 🏯 武士団 v18 — クイックスタート ダッシュボード

全体状況を一覧確認するエントリポイント。

| ノートブック | 内容 |
|---|---|
| `role_effectiveness.ipynb` | ロール別パフォーマンス・使用率 |
| `performance_metrics.ipynb` | レイテンシ・スループット分析 |
| `audit_analysis.ipynb` | YAML監査ログ詳細分析 |
| `skill_evolution_dashboard.ipynb` | スキル自動進化の状況 |

In [ ]:
import utils
from utils import qdf, scalar, ROLE_JA, PALETTE
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

## 1. システム概要

In [ ]:
# ── 基本集計 ──────────────────────────────────────────────────────
thread_count   = scalar("SELECT COUNT(*) FROM threads WHERE NOT is_archived")
msg_count      = scalar("SELECT COUNT(*) FROM messages")
user_msg_count = scalar("SELECT COUNT(*) FROM messages WHERE role='user'")
bot_msg_count  = scalar("SELECT COUNT(*) FROM messages WHERE role='assistant'")
route_count    = scalar("SELECT COUNT(*) FROM routing_decisions")
skill_cand     = scalar("SELECT COUNT(*) FROM skill_candidates")
skill_obs      = scalar("SELECT COUNT(*) FROM skill_observations")

print('=' * 40)
print('  武士団 v18  システム概要')
print('=' * 40)
print(f'  スレッド数 (アクティブ) : {thread_count:>6}')
print(f'  メッセージ総数          : {msg_count:>6}')
print(f'    └ ユーザー            : {user_msg_count:>6}')
print(f'    └ エージェント        : {bot_msg_count:>6}')
print(f'  ルーティング決定記録    : {route_count:>6}')
print(f'  スキル候補              : {skill_cand:>6}')
print(f'  スキル観察              : {skill_obs:>6}')
print('=' * 40)

## 2. 直近7日間のアクティビティ

In [ ]:
daily = qdf("""
    SELECT DATE(created_at) AS day,
           COUNT(*) FILTER (WHERE role='user')      AS user_msgs,
           COUNT(*) FILTER (WHERE role='assistant') AS bot_msgs
    FROM messages
    WHERE created_at >= NOW() - INTERVAL '7 days'
    GROUP BY DATE(created_at)
    ORDER BY day
""")

if utils.check_data(daily, '直近7日のメッセージ'):
    daily['day'] = pd.to_datetime(daily['day'])
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(daily['day'], daily['user_msgs'],  label='ユーザー',     color=PALETTE[1], alpha=0.8)
    ax.bar(daily['day'], daily['bot_msgs'],   label='エージェント', color=PALETTE[0], alpha=0.8,
           bottom=daily['user_msgs'])
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%m/%d'))
    ax.set_title('直近7日間 メッセージ数')
    ax.set_ylabel('メッセージ数')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f'  合計: ユーザー {daily["user_msgs"].sum()} 件 / エージェント {daily["bot_msgs"].sum()} 件')

## 3. ロール使用分布

In [ ]:
role_usage = qdf("""
    SELECT agent_role, COUNT(*) AS cnt
    FROM messages
    WHERE role='assistant' AND agent_role IS NOT NULL AND agent_role != ''
    GROUP BY agent_role
    ORDER BY cnt DESC
""")

if utils.check_data(role_usage, 'ロール使用データ'):
    role_usage['label'] = role_usage['agent_role'].map(
        lambda r: ROLE_JA.get(r, r)
    )
    colors = [utils.ROLE_COLOR.get(r, '#AAAAAA') for r in role_usage['agent_role']]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # 棒グラフ
    axes[0].barh(role_usage['label'], role_usage['cnt'], color=colors, edgecolor='white')
    axes[0].set_title('ロール別 応答数')
    axes[0].set_xlabel('応答数')
    for bar in axes[0].patches:
        axes[0].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                     f'{bar.get_width():.0f}', va='center', fontsize=9)

    # 円グラフ
    axes[1].pie(role_usage['cnt'], labels=role_usage['label'],
                colors=colors, autopct='%1.1f%%', startangle=90)
    axes[1].set_title('ロール別 使用割合')

    plt.suptitle('v18 ロール使用状況', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. 直近スレッド一覧

In [ ]:
recent = qdf("""
    SELECT t.id, t.title, t.tags,
           COUNT(m.id) AS msg_count,
           MAX(m.created_at) AS last_msg,
           STRING_AGG(DISTINCT m.agent_role, ', ') FILTER (WHERE m.agent_role != '') AS roles_used
    FROM threads t
    LEFT JOIN messages m ON m.thread_id = t.id
    WHERE NOT t.is_archived
    GROUP BY t.id, t.title, t.tags
    ORDER BY last_msg DESC NULLS LAST
    LIMIT 10
""")

if utils.check_data(recent, 'スレッドデータ'):
    recent['title_short'] = recent['title'].str[:40]
    recent['last_msg']    = pd.to_datetime(recent['last_msg']).dt.strftime('%m/%d %H:%M')
    display(recent[['title_short', 'msg_count', 'last_msg', 'roles_used']]
            .rename(columns={'title_short':'タイトル','msg_count':'メッセージ数',
                             'last_msg':'最終更新','roles_used':'使用ロール'}))

## 5. レスポンスタイム概要

In [ ]:
rt = qdf("""
    SELECT agent_role,
           ROUND(AVG(execution_time)::numeric, 0)  AS avg_ms,
           ROUND(MIN(execution_time)::numeric, 0)  AS min_ms,
           ROUND(MAX(execution_time)::numeric, 0)  AS max_ms,
           PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY execution_time) AS p95_ms,
           COUNT(*) AS n
    FROM messages
    WHERE role='assistant' AND execution_time > 0
      AND agent_role IS NOT NULL AND agent_role != ''
    GROUP BY agent_role
    HAVING COUNT(*) >= 1
    ORDER BY avg_ms DESC
""")

if utils.check_data(rt, 'レスポンスタイムデータ'):
    rt['役職'] = rt['agent_role'].map(lambda r: ROLE_JA.get(r, r))
    rt[['役職','avg_ms','min_ms','max_ms','p95_ms','n']].rename(
        columns={'avg_ms':'平均(ms)','min_ms':'最小(ms)','max_ms':'最大(ms)',
                 'p95_ms':'P95(ms)','n':'件数'}
    ).set_index('役職').pipe(display)